In [0]:
create table if not exists healthcare.silver.patients
using delta
as
with deduplicated as (
  select *,
    row_number() over (
      partition by mrn
      order by dbx_ingest_ts desc
    ) as rn
  from healthcare.bronze.patients
),
cleaned as (
  select
    try_cast(patient_id as int) as patient_id,
    mrn,
    initcap(trim(full_name)) as full_name,
    try_to_date(date_of_birth, 'yyyy/MM/dd') as date_of_birth,
    upper(gender) as gender,
    initcap(trim(city)) as city,
    blood_type,
    trim(phone) as phone,
    dbx_ingest_ts,
    dbx_source_file
  from deduplicated
  where rn = 1
)
select *
from cleaned
where patient_id is not null;


num_affected_rows,num_inserted_rows


In [0]:
--  drop table healthcare.silver.patients

In [0]:
select * from healthcare.silver.patients

patient_id,mrn,full_name,date_of_birth,gender,city,blood_type,phone,dbx_ingest_ts,dbx_source_file
1,MRN-00001,Aryan Maharaj,1968-08-03,M,Kolhapur,B-,1819600133,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
2,MRN-00002,Rushil Saini,1965-04-06,OTHER,Giridih,A-,265423511,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
3,MRN-00003,Gunbir Parmer,1982-01-27,M,Varanasi,A+,7816184959,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
4,MRN-00004,Ekaraj Bath,2025-04-04,OTHER,Ramagundam,B-,913164752553,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
5,MRN-00005,Ekbal Garg,2007-12-16,M,Ujjain,B+,6483503056,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
6,MRN-00006,Dominic Kakar,2012-06-14,M,Rajpur Sonarpur,O-,,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
7,MRN-00007,Meghana Shanker,1954-09-15,F,Kolhapur,O+,918849696532,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
8,MRN-00008,Gavin Batta,1950-03-08,OTHER,Kozhikode,AB+,,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
9,MRN-00009,Devansh Rajan,1987-03-14,OTHER,Dehri,O-,8018451462,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv
10,MRN-00010,Aarush Lall,2003-05-23,M,Guna,O+,914893252880,2026-04-15T13:21:44.360Z,patients_export_20260415_183022.csv


In [0]:
create table if not exists healthcare.silver.doctors
using delta
as
with deduplicated as (
  select *,
  row_number() over(
    partition by employee_code
    order by dbx_ingest_ts DESC ) as rn
  from healthcare.bronze.doctors
)
select 
cast(doctor_id as int) as doctor_id,
employee_code,
initcap(trim(full_name)) as full_name,
trim(department) as department,
cast(experience_years as int) as experience_years,
is_active,
dbx_ingest_ts,
dbx_source_file
from deduplicated
where rn = 1
and doctor_id is not null;


num_affected_rows,num_inserted_rows


In [0]:
select * from healthcare.silver.doctors 

doctor_id,employee_code,full_name,department,experience_years,is_active,dbx_ingest_ts,dbx_source_file
1,EMP-001,Balendra Singhal,Radiology,3,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
2,EMP-002,Nihal Sibal,Pathology,35,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
3,EMP-003,Jack Talwar,ICU,16,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
4,EMP-004,Riya Parikh,OPD,40,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
5,EMP-005,Dr. Tanvi Tailor,Other,28,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
6,EMP-006,Manbir Dasgupta,Emergency,15,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
7,EMP-007,Girik Jani,Radiology,38,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
8,EMP-008,Pranit Ganguly,Surgery,24,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
9,EMP-009,Simon Agrawal,Radiology,33,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv
10,EMP-010,Omaja Bose,Surgery,7,Y,2026-04-15T13:21:52.660Z,doctors_export_20260415_160031.csv


In [0]:
create table if not exists healthcare.silver.visits 
using delta 
as 
with deduplicated as (
  select *,
  row_number() over(
    partition by visit_id
    order by dbx_ingest_ts DESC ) as rn
  from healthcare.bronze.visits
)
select 
cast(visit_id as int) as visit_id,
cast(patient_id as int) as patient_id,
cast(doctor_id as int) as doctor_id,
try_to_date(visit_date, 'yyyy-MM-dd') as visit_date,
trim(department) as department,
chief_complaint,
case 
    when discharge_date = '' or discharge_date is null then null 
    else try_cast(discharge_date as date) 
end as discharge_date,
dbx_ingest_ts,
dbx_source_file
from deduplicated
where rn = 1 and visit_id is not null;


num_affected_rows,num_inserted_rows


In [0]:
select * from healthcare.silver.visits

visit_id,patient_id,doctor_id,visit_date,department,chief_complaint,discharge_date,dbx_ingest_ts,dbx_source_file
1,398,16,2024-09-15,OPD,Fatigue,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
10,326,43,2025-03-07,Pathology,Severe headache,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
100,170,14,2025-10-10,OPD,Nausea and vomiting,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
1000,467,46,2024-11-06,Pathology,Swollen legs,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
1001,414,15,2025-06-21,OPD,Skin rash,2025-06-23,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
1002,380,38,2025-04-28,OPD,Abdominal pain,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
1003,437,21,2025-04-23,Surgery,Chest pain,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
1004,434,25,2024-08-15,Surgery,Fatigue,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
1005,267,43,2025-06-07,Radiology,Abdominal pain,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv
1006,162,10,2024-09-02,Other,Dizziness,null,2026-04-15T13:22:00.725Z,visits_export_20260415_160031.csv


In [0]:
create or replace table healthcare.silver.treatments
using delta 
as 
with deduplicated as(
  select *,
    row_number() over (
      partition by treatment_id
      order by dbx_ingest_ts desc ) as rn
  from healthcare.bronze.treatments
)
select 
cast(treatment_id as int) as treatment_id,
cast(visit_id as int) as visit_id,
initcap(trim(treatment_name)) as treatment_name,
try_cast(cost as decimal(10,0)) as cost,
administered_by,
try_cast(treatment_date as date) as treatment_date,
dbx_ingest_ts,
dbx_source_file
from deduplicated 
where rn = 1 and treatment_id is not null;


num_affected_rows,num_inserted_rows


In [0]:
select * from healthcare.silver.treatments;

treatment_id,visit_id,treatment_name,cost,administered_by,treatment_date,dbx_ingest_ts,dbx_source_file
1,213,Insulin Therapy,35093,Caleb Bhavsar,2026-02-21,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
10,1012,Dialysis,3622,Jeet More,2024-10-05,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
100,1695,Colonoscopy,44263,Yutika Sachdev,2025-08-30,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
1000,316,Blood Transfusion,49051,Arjun Karpe,2025-08-08,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
1001,432,Iv Drip,35462,Bahadurjit Magar,2025-11-08,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
1002,1204,Physiotherapy,35362,Krishna Rattan,2024-07-15,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
1003,296,Biopsy,14248,Frado Bose,2024-06-01,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
1004,481,X-ray,38799,Anmol Aurora,2025-11-02,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
1005,551,X-ray,11277,Girish Raman,2024-09-02,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv
1006,676,Ecg,8016,Ekiya Sandal,2025-05-07,2026-04-25T18:32:08.397Z,treatments_export_20260425_235515.csv


In [0]:
create or replace table  healthcare.silver.claims
using delta 
as 
with dedpulicated as (
  select *,
    row_number() over(
      partition by claim_id,claim_date
      order by dbx_ingest_ts asc
    ) as rn
  from healthcare.bronze.claims  
)
select 
cast(claim_id as int) as claim_id,
cast(patient_id as int) as patient_id,
case 
    when visit_id = '' or visit_id is null then null
    else cast(visit_id as int)
  end as visit_id,
  try_cast(claim_amount as decimal(12,2)) as claim_amount,
  initcap(trim(insurer_name)) as insurer_name,
  trim(policy_number) as policy_number,
  trim(status) as status,
  try_to_date(claim_date, 'yyyy-MM-dd') as claim_date,
  try_to_date(settled_date, 'yyyy-MM-dd') as settled_date,
  dbx_ingest_ts,
  dbx_source_file
from dedpulicated
where rn = 1 and claim_id is not null;
select * from healthcare.silver.claims

claim_id,patient_id,visit_id,claim_amount,insurer_name,policy_number,status,claim_date,settled_date,dbx_ingest_ts,dbx_source_file
1,440,1313,58953.14,Tata Aig Health,POL-KM8019CK,Approved,2024-08-27,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
10,243,1868,149652.97,Care Health Insurance,POL-GV2661IR,Under Review,2025-06-08,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
100,403,950,49232.65,United India Insurance,POL-QP8310VI,Under Review,2026-01-02,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
1000,364,null,107671.04,Other,POL-FU3643NS,Under Review,2026-03-17,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
101,12,200,148599.81,Bajaj Allianz Health,POL-NL2845FQ,Pending,2025-08-02,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
102,460,1353,158701.86,Aditya Birla Health,POL-JF6821VY,Pending,2025-08-12,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
103,382,151,93539.18,Other,POL-BY7467ZC,Submitted,2025-09-16,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
104,127,1224,101948.85,Care Health Insurance,POL-AH9332TX,Approved,2024-12-07,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
105,384,null,135974.48,New India Assurance,POL-UQ1746NU,Rejected,2025-10-23,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv
106,31,354,170161.59,Niva Bupa Health,POL-KC8191FS,Rejected,2025-07-30,null,2026-04-25T18:32:13.194Z,claims_export_20260425_235321.csv


In [0]:
-- drop table healthcare.silver.claims;

Qurantine tables 

In [0]:
create or replace table healthcare.silver.quarantine_patients
using delta
as
with deduplicated as (
  select *,
  row_number() over(
    partition by mrn order by dbx_ingest_ts desc ) as rn
    from healthcare.silver.patients
  )
select 
patient_id ,
mrn,
trim(full_name) as full_name,
date_of_birth,
gender,
city,
blood_type,
phone,
dbx_ingest_ts,
dbx_source_file,
    CASE
        WHEN patient_id IS NULL
            THEN 'NULL patient_id'
        WHEN TRY_CAST(date_of_birth AS DATE) IS NULL
            THEN 'Invalid date_of_birth format'
    END                                 AS rejection_reason
FROM deduplicated
WHERE rn > 1;
select * from healthcare.silver.quarantine_patients;


patient_id,mrn,full_name,date_of_birth,gender,city,blood_type,phone,dbx_ingest_ts,dbx_source_file,rejection_reason
